# STRATA — Cuaderno de **experimentos** (no canónico)

Exploraciones metodológicas que **respaldan** las decisiones del cuaderno canónico
(`strata_canonical.ipynb`, SPY con HMM de 3 estados y gate $\tau=0.5$) pero que, por alcance, no
viven en él. Cada experimento tiene su pre-registro en `BITACORA.md` y su script reproducible en
`experiments/`. Aquí solo se cargan los resultados (`outputs/experiments/*.json`) y se interpretan.

**Hilo conductor — la elección del número de regímenes $K$:** un binario ($K=2$) da *más*
accuracy y Sharpe que $K=3$ en el OOS de SPY. Estas secciones demuestran, sin mirar el futuro de
forma circular, que esa ventaja **no es real** (es cabalgar el drift alcista) y que **$K=3$ es la
elección correcta**.

In [1]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_ROOT = Path.cwd() if (Path.cwd() / "config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(_ROOT)); os.chdir(_ROOT)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


def load(name):
    return json.load(open(f"outputs/experiments/{name}.json"))

## E1. Selección de $K$ por verosimilitud fuera de muestra (sin OOS)

El criterio honesto para decidir cuántos regímenes describen los datos no es el P&L de una
ventana, sino la **verosimilitud fuera de muestra** dentro de la calibración (validación temporal
*expanding-window*, 2000–2024, sin tocar el OOS 2024-10+): ¿cada estado extra mejora la
descripción de datos *no vistos*, o sobreajusta? Lo evaluamos por activo
(`experiments/k_selection_panel.py`).

In [2]:
sel = load("k_selection_panel")
df = pd.DataFrame(sel["per_asset"])
tab = df[["ticker", "heldout_LL_K2", "heldout_LL_K3", "delta_LL_K3_minus_K2", "k_elegido", "vol_anual_calib"]]
tab = tab.rename(columns={"delta_LL_K3_minus_K2": "ΔLL(K3−K2)", "k_elegido": "K elegido", "vol_anual_calib": "vol calib"})
display(tab.set_index("ticker"))
print(f"K=3 elegido en {sel['n_k3']}/{sel['n_k3'] + sel['n_k2']} activos; K=2 en {sel['n_k2']}.")
print(f"ΔLL(K3−K2) > 0 en TODOS: el tercer estado mejora la descripción fuera de muestra en cada activo.")
print(f"Correlación vol↔ventaja-de-K3: ρ={sel['spearman_vol_vs_deltaLL']['rho']:+.2f} "
      f"(p={sel['spearman_vol_vs_deltaLL']['p']:.2f}) → la volatilidad NO decide K.")

,heldout_LL_K2,heldout_LL_K3,ΔLL(K3−K2),K elegido,vol calib
ticker,,,,,
SPY,-1.693,-1.301,0.392,3,0.194
NVDA,-1.746,-1.432,0.314,3,0.594
BAC,-0.894,-0.773,0.122,3,0.438
TSLA,-2.765,-2.360,0.405,3,0.563
XLE,-2.174,-1.660,0.514,3,0.292
UNG,-3.088,-2.983,0.105,3,0.480
MSTR,-1.462,-1.234,0.228,3,0.731
SMCI,-2.569,-2.173,0.396,3,0.577
ROKU,-2.278,-2.193,0.084,3,0.755


K=3 elegido en 10/10 activos; K=2 en 0.
ΔLL(K3−K2) > 0 en TODOS: el tercer estado mejora la descripción fuera de muestra en cada activo.
Correlación vol↔ventaja-de-K3: ρ=-0.14 (p=0.70) → la volatilidad NO decide K.


**Lectura E1.** Por verosimilitud fuera de muestra, **$K=3$ bate a $K=2$ en los 10/10 activos**,
con holgura y con independencia de la volatilidad ($\rho\approx0$). El tercer estado mejora la
descripción de datos no vistos en cada activo —no es relleno—. Cautela: este test compara solo
$K=2$ vs $K=3$; la LL es monótona creciente en $K$ (gaussiana mal especificada), así que **no
selecciona $K=3$ como óptimo global** —$K\ge4$ se descarta por interpretabilidad, no por LL—. Lo
que queda firme: el "$K=2$ mejor" del trading OOS NO proviene de la estructura de los datos.

## E2. ¿Y elegir $K$ por activo con un criterio direccional? (exploración honesta)

Pregunta natural: ¿se puede elegir $K$ por activo, *a pasado*, con un criterio alineado al uso de
STRATA (la **dirección**, no la densidad)? Criterio ex-ante: el $K$ con mejor **acierto direccional
del régimen fuera de muestra en calibración** (`experiments/k_selection_directional.py`). Lo
**validamos** comparando el $K$ elegido en calibración con el $K$ que mejor rinde en el OOS
(diagnóstico, no selección; `experiments/k_per_asset_directional.py`).

In [3]:
from scipy.stats import binomtest

pad = load("k_per_asset_directional"); agg = pad["aggregate"]
df2 = pd.DataFrame(pad["per_asset"])
tab2 = df2[["ticker", "calib_K", "oos_best_K", "match", "oos_sharpe_K2", "oos_sharpe_K3"]]
display(tab2.set_index("ticker"))

n, k = agg["n_assets"], agg["n_match_calibK_vs_oosbestK"]
p_sign = binomtest(k, n, 0.5, alternative="greater").pvalue
print(f"Concordancia K(calibración) vs K(mejor-OOS): {k}/{n}  →  sign test vs azar p={p_sign:.3f}")
print(f"Cartera K-por-activo: Sharpe OOS medio {agg['mean_oos_sharpe_perasset']:+.3f}  "
      f"vs K=3 fijo {agg['mean_oos_sharpe_fixedK3']:+.3f}  (Diebold-Mariano p={agg['dm_perasset_vs_k3_p']:.2f})")

,calib_K,oos_best_K,match,oos_sharpe_K2,oos_sharpe_K3
ticker,,,,,
SPY,2,2,True,1.329,0.672
NVDA,3,3,True,0.531,0.672
BAC,2,3,False,0.289,1.079
TSLA,3,3,True,-0.714,-0.342
XLE,2,2,True,1.449,0.835
UNG,3,2,False,0.676,0.325
MSTR,3,3,True,-0.478,-0.100
SMCI,2,2,True,0.685,0.266
ROKU,2,3,False,-0.234,0.361


Concordancia K(calibración) vs K(mejor-OOS): 7/10  →  sign test vs azar p=0.172
Cartera K-por-activo: Sharpe OOS medio +0.461  vs K=3 fijo +0.430  (Diebold-Mariano p=0.60)


**Lectura E2 (honesta).** El criterio direccional tiene *algo* de valor predictivo
(concordancia 7/10, mejor que el 1/10 de criterios ingenuos), pero **no supera al azar de forma
significativa** (sign test $p\approx0.17$, $n=10$) y la cartera K-por-activo **no bate
significativamente** a $K=3$ fijo (Diebold-Mariano $p\approx0.60$). Con $n=10$ y una sola ventana
OOS, los desajustes son indistinguibles del ruido. Conclusión: elegir $K$ por activo es una
extensión **prometedora pero no demostrada**; el canónico usa $K=3$ fijo (E1) y esto queda como
trabajo futuro a validar en un panel mayor.

## E3. ¿Cuándo bate el meta-learner (M10) a la regla a mano (M8)? — el test del drift

Es la pieza que cierra *por qué* $K=2$ parecía mejor. Si la ventaja de $M8$ fuera destreza de
régimen, batiría al meta-learner $M10$ con independencia del drift del activo; si es cabalgar el
drift, $M10$ (adaptativo) debería ganarle en activos que **caen**. Comparamos $M10$ vs $M8$
**para K=2 Y K=3** en los 10 activos y medimos $\rho(\text{drift}, M10-M8)$ por cada $K$
(`experiments/drift_test_k2k3.py`). Así vemos si la abstención de $K=3$ reduce la dependencia del
drift respecto a $K=2$ —el test previo solo corría $K=2$ y no podía responderlo (autogol corregido)—.

In [4]:
dr = load("drift_test_k2k3")  # K=2 Y K=3 sobre los 10 activos (corrige el autogol del test previo a K=2)
for K in ("2", "3"):
    d = dr[K]
    t = pd.DataFrame(d["per_asset"]).sort_values("drift_oos", ascending=False)[
        ["ticker", "drift_oos", "M8", "M10", "M10_minus_M8"]]
    print(f"═══ K={K}:  ρ(drift, M10−M8) = {d['rho_drift_vs_M10minusM8']:+.2f}  (p={d['p']:.2f}, n={len(t)})  "
          f"| M8 bate a M10 en {d['n_M8_beats_M10']}/{len(t)} activos")
    display(t.set_index("ticker"))
rho2, rho3 = dr["2"]["rho_drift_vs_M10minusM8"], dr["3"]["rho_drift_vs_M10minusM8"]
print(f"\nComparación: ρ_K2={rho2:+.2f}  vs  ρ_K3={rho3:+.2f}")
if rho2 < -0.4 and abs(rho3) < abs(rho2) - 0.25:
    print("→ K=2 es claramente más CONDICIONAL al drift que K=3: la abstención de K=3 reduce el drift-riding.")
elif rho3 < -0.4:
    print("→ K=3 TAMBIÉN es condicional al drift (ρ_K3 fuerte y negativo): cabalga el drift menos que K=2 pero")
    print("  no es 'puro supervisor'. Honesto: la diferencia es de GRADO, no categórica.")
else:
    print("→ Lectura matizada: ver magnitudes; con n=10 y p no significativos, es sugerente, no concluyente.")

═══ K=2:  ρ(drift, M10−M8) = -0.36  (p=0.31, n=10)  | M8 bate a M10 en 6/10 activos


,drift_oos,M8,M10,M10_minus_M8
ticker,,,,
NVDA,0.380,0.531,-1.479,-2.010
ROKU,0.340,-0.234,0.941,1.175
TSLA,0.330,-0.714,-2.278,-1.565
XLE,0.190,1.449,1.358,-0.092
SPY,0.160,1.329,0.471,-0.858
BAC,0.160,0.289,0.160,-0.129
MSTR,0.090,-0.478,-0.413,0.065
MARA,-0.120,-0.016,0.698,0.714
SMCI,-0.140,0.685,0.349,-0.336


═══ K=3:  ρ(drift, M10−M8) = -0.70  (p=0.02, n=10)  | M8 bate a M10 en 9/10 activos


,drift_oos,M8,M10,M10_minus_M8
ticker,,,,
NVDA,0.380,0.672,-2.557,-3.228
ROKU,0.340,0.361,-0.147,-0.508
TSLA,0.330,-0.342,-1.912,-1.570
XLE,0.190,0.835,0.189,-0.646
SPY,0.160,0.672,0.642,-0.030
BAC,0.160,1.079,-0.421,-1.500
MSTR,0.090,-0.100,-0.851,-0.751
MARA,-0.120,0.536,0.493,-0.043
SMCI,-0.140,0.266,0.072,-0.194



Comparación: ρ_K2=-0.36  vs  ρ_K3=-0.70
→ K=3 TAMBIÉN es condicional al drift (ρ_K3 fuerte y negativo): cabalga el drift menos que K=2 pero
  no es 'puro supervisor'. Honesto: la diferencia es de GRADO, no categórica.


**Lectura E3 (honesta, corrige el test previo).** El test anterior solo corría $K=2$ —no podía
decir nada de $K=3$—. Aquí comparamos ambos en los 10 activos. $\rho<0$ significa que $M8$ bate al
meta-learner $M10$ **solo en activos alcistas** (cabalga el drift). La comparación $\rho_{K2}$ vs
$\rho_{K3}$ dice si la abstención de $K=3$ **reduce** esa dependencia del drift respecto a $K=2$.

Importante: con $n=10$ y $p$ no significativos, esto es **sugerente, no una prueba**. Mide un
*grado* de drift-riding, no una dicotomía. El cierre robusto de esta cuestión es la **validación
multi-ventana / walk-forward** (2008, 2020, 2022 de la calibración como pseudo-OOS), **pendiente**:
sin ella, "$K=3$ supervisa y $K=2$ cabalga" es una hipótesis bien fundada, no un resultado cerrado.

### Síntesis: por qué $K=3$ (decisión razonada, con sus límites)

1. **E1** — la verosimilitud fuera de muestra bate a $K=2$ en los 10 activos (la LL pediría más
   estados, pero $K\ge4$ no es interpretable; dentro de $\{2,3\}$ gana 3).
2. **E2** — elegir $K$ por activo no está estadísticamente respaldado ($7/10$, $p=0.17$).
3. **E3** — la ventaja nominal de $K=2$ es (sugerentemente) cabalgar el drift; $K=3$ se abstiene.
4. **Mecanismo** — el estado **Estrés = abstención** hace de STRATA un *supervisor*, no un
   *cabalga-drifts*: $K=3$ interviene selectivamente (~49 % de días), $K=2$ por defecto (~75 %).

El canónico adopta $K=3$ deliberadamente. **Límite reconocido:** todo esto vive en una única
ventana OOS alcista; la robustez multi-ventana es trabajo pendiente.